Author: Giovanni Squillero <[giovanni.squillero@polito.it](giovanni.squillero@polito.it)>  
Copyright © 2026 [Giovanni Squillero](https://squillero.github.io/) / [Politecnico di Torino](https://www.polito.it)  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE) for details.  

In [30]:
RANDOM_SEED = 10
SIZE = 4

Let's solve the $(s^2-1)$-puzzle

In [31]:
from upq.upq import UPQ

In [32]:
import logging
from random import seed, choice, shuffle
from collections.abc import Callable

logging.getLogger().setLevel(logging.INFO)

In [33]:
import numpy as np

In [34]:
class State:
    def __init__(self, data: np.ndarray):
        self._data = data.copy()
        self._data.flags.writeable = False

    def __hash__(self):
        return hash(bytes(self._data))

    def __eq__(self, other):
        return bytes(self._data) == bytes(other._data)

    def __lt__(self, other):
        return bytes(self._data) < bytes(other._data)

    def __str__(self):
        return str(self._data)

    def __repr__(self):
        return repr(self._data)

    @property
    def data(self):
        return self._data

    def copy_data(self):
        return self._data.copy()

# Search Algorithms

In [35]:
def search(
    initial_state: State,
    goal_test: Callable,
    parent_state: dict,
    state_cost: dict,
    priority_function: Callable,
    unit_cost: Callable,
):
    frontier = UPQ()
    parent_state.clear()
    state_cost.clear()

    state = initial_state
    parent_state[state] = None
    state_cost[state] = 0

    while state is not None and not goal_test(state):
        for a in possible_actions(state):
            new_state = result(state, a)
            cost = unit_cost(a)
            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                state_cost[new_state] = state_cost[state] + cost
                frontier.push_update(new_state, priority_function(new_state))
                logging.debug(f"Added new node to frontier (cost={state_cost[new_state]})")
            elif new_state in frontier and state_cost[new_state] > state_cost[state] + cost:
                old_cost = state_cost[new_state]
                parent_state[new_state] = state
                state_cost[new_state] = state_cost[state] + cost
                frontier.push_update(new_state, priority_function(new_state))
                logging.debug(
                    f"Updated node cost in frontier: {old_cost} -> {state_cost[new_state]}"
                )
        if frontier:
            state, _ = frontier.pop()
        else:
            state = None

    path = list()
    s = state
    while s:
        path.append(s.copy_data())
        s = parent_state[s]

    logging.info(f"Found a solution in {len(path):,} steps; visited {len(state_cost):,} states")
    return list(reversed(path))

## Graph search for the the *n*-puzzle

In [36]:
seed(RANDOM_SEED)

In [37]:
GOAL = State(np.array(list(range(1, SIZE**2)) + [0]).reshape((SIZE, SIZE)))
logging.info(f"Goal:\n{GOAL}")


def goal_test(state):
    return state == GOAL

INFO:root:Goal:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15  0]]


In [39]:
# (R, C) -> UP / RIGHT / DOWN / LEFT
MOVES = [np.array(_) for _ in [(-1, 0), (0, +1), (+1, 0), (0, -1)]]

In [40]:
def find_empty_space(board: np.ndarray):
    t = np.where(board == 0)
    return np.array([t[0][0], t[1][0]])


def is_valid(board: np.ndarray, action):
    return all(0 <= (find_empty_space(board) + action)[i] < board.shape[i] for i in [0, 1])


def possible_actions(state: State):
    return (m for m in MOVES if is_valid(state._data, m))

In [41]:
def result(state, action):
    board = state.copy_data()
    space = find_empty_space(board)
    pos = space + action
    board[space[0], space[1]] = board[pos[0], pos[1]]
    board[pos[0], pos[1]] = 0
    return State(board)

In [42]:
INITIAL_STATE = GOAL
for r in range(5_000):
    INITIAL_STATE = result(INITIAL_STATE, choice(list(possible_actions(INITIAL_STATE))))
INITIAL_STATE

array([[ 5,  6,  0,  7],
       [10,  4,  8, 12],
       [ 9,  2, 15, 14],
       [11,  1, 13,  3]])

## Breadth-First

In [43]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: len(state_cost),
    unit_cost=lambda a: 1,
)

KeyboardInterrupt: 

## Depth-First

In [44]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: -len(state_cost),
    unit_cost=lambda a: 1,
)

KeyboardInterrupt: 

## Gready Best-First

In [45]:
parent_state = dict()
state_cost = dict()


def h(state):
    return np.sum((state._data != GOAL._data) & (state._data > 0))


final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: h(s),
    unit_cost=lambda a: 1,
)

INFO:root:Found a solution in 265 steps; visited 8,196 states


## A*

In [46]:
parent_state = dict()
state_cost = dict()


def h(state):
    return np.sum((state._data != GOAL._data) & (state._data > 0))


final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
)

KeyboardInterrupt: 

In [ ]:
element = 4
data = np.array(GOAL)
t = np.where(data == element)
np.array([t[0][0], t[1][0]])


AttributeError: 'int' object has no attribute '_data'

In [54]:
GOAL == 8

AttributeError: 'int' object has no attribute '_data'